# Module 2 — Goal Planning

**Key concepts:**
- **Time Value of Money** — ₹50L today is NOT ₹50L in 15 years. Inflation erodes purchasing power.
- **Future Value** — we first inflate the goal to what it will actually cost.
- **Required SIP** — we work backwards from future value to the monthly investment needed.
- **Feasibility** — how much of the required SIP can your current savings cover?

Supports multiple simultaneous goals with different horizons and priorities.

In [ ]:
# ── Setup (Colab) ────────────────────────────────────────────────
!pip install -q numpy pandas plotly
import os
if not os.path.exists('robo-advisor'):
    !git clone https://github.com/parinmody30/robo-advisor.git
else:
    !git -C robo-advisor pull

import sys
sys.path.insert(0, 'robo-advisor/src')
DATA_DIR = 'robo-advisor/data'
print('Setup complete.')

In [ ]:
import json
from goal_planner import plan_goals, summary_table

# Load risk profile from Module 1
try:
    with open(f'{DATA_DIR}/risk_profile.json') as f:
        profile = json.load(f)
    expected_return = profile['target_return']
    persona = profile['persona']
    print(f'Loaded profile: {persona} | Expected return: {expected_return*100:.0f}% p.a.')
except FileNotFoundError:
    expected_return = 0.12
    persona = 'Balanced'
    print('No saved profile — using 12% return assumption.')

## 2A. Define your goals
Edit the list below. Each goal needs: `name`, `target_amount` (today's ₹), `years_to_goal`, `priority` (1=most important).

In [ ]:
goals = [
    {
        "name":           "Retirement Corpus",
        "target_amount":  10_000_000,   # ₹1 Crore in today's money
        "years_to_goal":  30,
        "priority":       1,
        "inflation_rate": 0.06,
    },
    {
        "name":           "House Down Payment",
        "target_amount":  2_000_000,    # ₹20L in today's money
        "years_to_goal":  7,
        "priority":       2,
        "inflation_rate": 0.07,         # real estate inflates faster
    },
    {
        "name":           "Child's Education",
        "target_amount":  3_000_000,    # ₹30L in today's money
        "years_to_goal":  15,
        "priority":       3,
        "inflation_rate": 0.08,         # education inflation ~8%
    },
]

# Your current monthly savings available for investing
monthly_savings = 25_000   # ₹25,000/month

## 2B. Run the planner

In [ ]:
planned = plan_goals(goals, monthly_savings, expected_return)

import pandas as pd
df = pd.DataFrame(summary_table(planned))
print(df.to_string(index=False))

## 2C. Visualise — SIP required vs your savings

In [ ]:
import plotly.graph_objects as go

names = [g.name for g in planned]
sips  = [g.monthly_sip for g in planned]
feas  = [g.feasibility_pct for g in planned]
colors = ['#2E7D32' if f >= 100 else '#E65100' if f < 50 else '#F9A825' for f in feas]

fig = go.Figure()
fig.add_trace(go.Bar(name='Required SIP (₹/mo)', x=names, y=sips,
                     marker_color=colors,
                     text=[f'₹{v:,.0f}' for v in sips], textposition='outside'))
fig.add_hline(y=monthly_savings, line_dash='dash', line_color='#1565C0',
              annotation_text=f'Your savings: ₹{monthly_savings:,}/mo',
              annotation_position='bottom right')
fig.update_layout(title='Monthly SIP Required per Goal',
                  yaxis_title='₹ / month', template='plotly_white', height=450)
fig.show()

## 2D. Visualise — inflation impact (today vs future value)

In [ ]:
fig2 = go.Figure()
fig2.add_trace(go.Bar(name="Today's value",
                      x=names, y=[g.target_amount for g in planned],
                      marker_color='#42A5F5'))
fig2.add_trace(go.Bar(name='Future value (inflation-adjusted)',
                      x=names, y=[g.future_value for g in planned],
                      marker_color='#EF5350'))
fig2.update_layout(barmode='group',
                   title='Inflation Impact: What Your Goals Will Actually Cost',
                   yaxis_title='₹', template='plotly_white', height=450)
fig2.show()

## 2E. Save goals for downstream modules

In [ ]:
import json, os
os.makedirs(DATA_DIR, exist_ok=True)

goals_out = [g.__dict__ for g in planned]
with open(f'{DATA_DIR}/goals.json', 'w') as f:
    json.dump({'monthly_savings': monthly_savings, 'goals': goals_out}, f, indent=2)
print(f'Goals saved to {DATA_DIR}/goals.json')

## Theory note

**Why inflation-adjust?** Most people set a goal of '₹1 Crore for retirement' without realising that ₹1 Crore in 30 years (at 6% inflation) has the purchasing power of roughly ₹17L today. The future value calculation forces honesty about the actual target.

**Why SIP over lumpsum?** Rupee-cost averaging — investing a fixed amount monthly means you automatically buy more units when markets are low and fewer when high. This reduces sequence-of-returns risk, especially important for long horizons.

**Feasibility flag:** If required SIP > monthly savings, the system flags the gap. The user has three levers: increase savings, extend the horizon, or reduce the target amount.